In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import seaborn as sns

In [ ]:
df = sns.load_dataset('iris')

In [ ]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
X = df.drop(columns=['species'])
y = df['species']

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model_knn = KNeighborsClassifier()

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
classifier = GridSearchCV((model_knn), {
    'n_neighbors':[5,7,9,13],
    'metric':['euclidean','manhattan','minkowski']
},cv=5,return_train_score=False)

In [ ]:
classifier.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'metric': ['euclidean', 'manhattan', 'minkowski'],
                         'n_neighbors': [5, 7, 9, 13]})

In [ ]:
results = pd.DataFrame(classifier.cv_results_)

In [ ]:
results.shape

(12, 15)

In [ ]:
results[['mean_test_score', 'param_metric', 'param_n_neighbors']]

,mean_test_score,param_metric,param_n_neighbors
0,0.925000,euclidean,5
1,0.941667,euclidean,7
2,0.941667,euclidean,9
3,0.950000,euclidean,13
4,0.950000,manhattan,5
5,0.950000,manhattan,7
6,0.958333,manhattan,9
7,0.933333,manhattan,13
8,0.925000,minkowski,5
9,0.941667,minkowski,7


In [ ]:
classifier.best_params_

{'metric': 'manhattan', 'n_neighbors': 9}

In [ ]:
best_model = classifier.best_estimator_

In [ ]:
best_model.score(X_test_scaled,y_test)

1.0

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': [5, 7, 9, 13],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski']
}

grid = GridSearchCV(pipeline, param_grid, cv=5)

grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('knn', KNeighborsClassifier())]),
             param_grid={'knn__metric': ['euclidean', 'manhattan', 'minkowski'],
                         'knn__n_neighbors': [5, 7, 9, 13]})

In [ ]:
from sklearn.svm import SVC

In [ ]:
model_svm = SVC(gamma='auto')

In [ ]:
classifier_svm = GridSearchCV((model_svm), {
    'C':[1,10,20,30],
    'kernel':['rbf', 'linear']
}, cv=5,return_train_score=False)

In [ ]:
classifier_svm.fit(X_test_scaled, y_test)

GridSearchCV(cv=5, estimator=SVC(gamma='auto'),
             param_grid={'C': [1, 10, 20, 30], 'kernel': ['rbf', 'linear']})

In [ ]:
results = pd.DataFrame(classifier_svm.cv_results_)

In [ ]:
results[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.933333
1,1,linear,0.933333
2,10,rbf,0.966667
3,10,linear,0.933333
4,20,rbf,0.966667
5,20,linear,0.933333
6,30,rbf,0.966667
7,30,linear,0.933333


In [ ]:
classifier_svm.best_params_

{'C': 10, 'kernel': 'rbf'}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# classifier_r = RandomizedSearchCV((model_svm))

In [ ]:
classifier_r = RandomizedSearchCV((model_svm), {
    'C':[1,10,20,30],
    'kernel':['rbf', 'linear']
}, n_iter=4,cv=5,return_train_score=False)

In [ ]:
classifier_r.fit(X_train_scaled, y_train)

RandomizedSearchCV(cv=5, estimator=SVC(gamma='auto'), n_iter=4,
                   param_distributions={'C': [1, 10, 20, 30],
                                        'kernel': ['rbf', 'linear']})

In [ ]:
results = pd.DataFrame(classifier_r.cv_results_)

In [ ]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kernel,param_C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002012,0.000770,0.001016,0.000070,linear,1,"{'kernel': 'linear', 'C': 1}",0.958333,1.000000,0.833333,1.000000,0.958333,0.950000,0.061237,1
1,0.001323,0.000043,0.000907,0.000022,rbf,1,"{'kernel': 'rbf', 'C': 1}",0.958333,1.000000,0.833333,1.000000,0.958333,0.950000,0.061237,1
2,0.001243,0.000058,0.000850,0.000007,linear,30,"{'kernel': 'linear', 'C': 30}",0.958333,0.958333,0.833333,0.958333,1.000000,0.941667,0.056519,3
3,0.001285,0.000039,0.000867,0.000010,rbf,10,"{'kernel': 'rbf', 'C': 10}",0.958333,0.958333,0.833333,1.000000,0.958333,0.941667,0.056519,4


In [ ]:
classifier_r.best_params_

{'kernel': 'linear', 'C': 1}

In [ ]:
best_model_svm = classifier_r.best_estimator_

In [ ]:
best_model_svm.score(X_test_scaled, y_test)

0.9666666666666667